# 143 SGD code -- mini-batch gradient descent

Lean PyTorch loop for **mini-batch GD** (what DL usually calls SGD).

Companion lectures: `141_stochastic_gradient_descent.py`, `142_SGD_training.py`

| Term | Meaning |
|------|---------|
| `w` | model parameters (a **condition** for the loss) |
| `L(... ; w)` | full-batch loss on all data, given `w` |
| mini-batch loss | loss on `b` rows -- builds the gradient for one step |
| gradient step | one `optimizer.step()` |


In [1]:
import torch
import torch.nn as nn

torch.manual_seed(0)

N, D, B, LR, EPOCHS = 512, 4, 32, 0.05, 20

X = torch.randn(N, D)
w_true = torch.tensor([1.4, -0.8, 0.55, 0.2])
y = X @ w_true + 0.3 * torch.randn(N)

model = nn.Linear(D, 1, bias=False)
opt = torch.optim.SGD(model.parameters(), lr=LR)
loss_fn = nn.MSELoss()

@torch.no_grad()
def full_batch_loss():
    """L(... ; w): mean loss on ALL N examples."""
    return loss_fn(model(X).squeeze(), y).item()


## Training loop

Each gradient step:
1. sample `b` rows
2. compute **mini-batch loss**
3. `backward` + `step`

`L(... ; w)` is evaluated only to monitor progress (not needed to take the step).


In [2]:
history = [full_batch_loss()]

for epoch in range(EPOCHS):
    perm = torch.randperm(N)
    for i in range(0, N, B):
        idx = perm[i : i + B]
        loss = loss_fn(model(X[idx]).squeeze(), y[idx])  # mini-batch loss
        opt.zero_grad()
        loss.backward()
        opt.step()                                       # gradient step
    history.append(full_batch_loss())
    print(f"epoch {epoch + 1:02d}  L(... ; w) = {history[-1]:.4f}")

print("learned w:", model.weight.detach().squeeze().tolist())
print("true    w:", w_true.tolist())


epoch 01  L(... ; w) = 0.2140
epoch 02  L(... ; w) = 0.0875
epoch 03  L(... ; w) = 0.0822
epoch 04  L(... ; w) = 0.0820
epoch 05  L(... ; w) = 0.0822
epoch 06  L(... ; w) = 0.0820
epoch 07  L(... ; w) = 0.0820
epoch 08  L(... ; w) = 0.0821
epoch 09  L(... ; w) = 0.0820
epoch 10  L(... ; w) = 0.0821
epoch 11  L(... ; w) = 0.0821
epoch 12  L(... ; w) = 0.0821
epoch 13  L(... ; w) = 0.0820
epoch 14  L(... ; w) = 0.0821
epoch 15  L(... ; w) = 0.0820
epoch 16  L(... ; w) = 0.0821
epoch 17  L(... ; w) = 0.0821
epoch 18  L(... ; w) = 0.0821
epoch 19  L(... ; w) = 0.0820
epoch 20  L(... ; w) = 0.0820
learned w: [1.410564661026001, -0.8082336783409119, 0.5287443399429321, 0.2095191776752472]
true    w: [1.399999976158142, -0.800000011920929, 0.550000011920929, 0.20000000298023224]
